In [ ]:
import glob

import xarray as xr

da_instant = glob.glob("../data/raw/**/*instant.nc")
da_accum = glob.glob("../data/raw/**/*accum.nc")

print(da_accum)
print(da_instant)

In [ ]:
da_instant = xr.open_mfdataset(da_instant, chunks='auto')
da_accum = xr.open_mfdataset(da_accum, chunks='auto')

In [ ]:
# checking experiment version (expver)
df_instant_expver = da_instant.expver.to_dataframe()
df_accum_expver = da_accum.expver.to_dataframe()

print(df_instant_expver.head())
print(df_accum_expver.head())

# check unqiue values for expver
df_instant_expver.expver.nunique()

In [ ]:
df_accum_expver.expver.nunique()

Its safe to drop expver here, since the values are all `1`.

In [ ]:
da_instant = da_instant.drop_vars("expver")
da_accum = da_accum.drop_vars("expver")

Transform `total precipitation`, as it is accumulated

In [ ]:

tp_daily = da_accum.resample(valid_time="1D").last()

In [ ]:
tp_daily

Resample instant variables also

In [ ]:
instant_daily = da_instant.resample(valid_time="1D").mean()
instant_daily

In [ ]:
ds_daily = xr.merge([
    instant_daily,
    tp_daily,
])

In [ ]:
ds_daily

In [1]:
import xarray as xr

ds = xr.open_dataset("../data/interim/combined_2020-2024_daily.nc", chunks="auto")
ds

<xarray.Dataset> Size: 535MB
Dimensions:     (valid_time: 1827, latitude: 121, longitude: 121)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 15kB 2020-01-01 ... 2024-12-31
  * latitude    (latitude) float64 968B 35.0 34.75 34.5 34.25 ... 5.5 5.25 5.0
  * longitude   (longitude) float64 968B 65.0 65.25 65.5 ... 94.5 94.75 95.0
    number      int64 8B ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 107MB dask.array<chunksize=(1827, 121, 121), meta=np.ndarray>
    u10         (valid_time, latitude, longitude) float32 107MB dask.array<chunksize=(1827, 121, 121), meta=np.ndarray>
    v10         (valid_time, latitude, longitude) float32 107MB dask.array<chunksize=(1827, 121, 121), meta=np.ndarray>
    d2m         (valid_time, latitude, longitude) float32 107MB dask.array<chunksize=(1827, 121, 121), meta=np.ndarray>
    tp          (valid_time, latitude, longitude) float32 107MB dask.array<chunksize=(1827, 121, 121), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-18T20:14 GRIB to CDM+CF via cfgrib-0.9.1...

In [2]:
ds.drop_vars("number")

<xarray.Dataset> Size: 535MB
Dimensions:     (valid_time: 1827, latitude: 121, longitude: 121)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 15kB 2020-01-01 ... 2024-12-31
  * latitude    (latitude) float64 968B 35.0 34.75 34.5 34.25 ... 5.5 5.25 5.0
  * longitude   (longitude) float64 968B 65.0 65.25 65.5 ... 94.5 94.75 95.0
Data variables:
    t2m         (valid_time, latitude, longitude) float32 107MB dask.array<chunksize=(1827, 121, 121), meta=np.ndarray>
    u10         (valid_time, latitude, longitude) float32 107MB dask.array<chunksize=(1827, 121, 121), meta=np.ndarray>
    v10         (valid_time, latitude, longitude) float32 107MB dask.array<chunksize=(1827, 121, 121), meta=np.ndarray>
    d2m         (valid_time, latitude, longitude) float32 107MB dask.array<chunksize=(1827, 121, 121), meta=np.ndarray>
    tp          (valid_time, latitude, longitude) float32 107MB dask.array<chunksize=(1827, 121, 121), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-18T20:14 GRIB to CDM+CF via cfgrib-0.9.1...

In [3]:
combined_df = ds.to_dataframe()

In [5]:
del combined_df["number"]

In [6]:
combined_df

t2m       u10       v10         d2m  \
valid_time latitude longitude                                               
2020-01-01 35.0     65.00      265.666962 -1.664091  2.675078  263.734283   
                    65.25      264.842896 -1.501531  1.938975  262.880920   
                    65.50      264.346039 -1.531428  1.486827  262.441620   
                    65.75      263.709320 -1.530377  1.263945  261.992859   
                    66.00      262.236359 -1.195341  1.411707  260.940552   
...                                   ...       ...       ...         ...   
2024-12-31 5.0      94.00      299.804138 -2.557239 -0.341172  296.730347   
                    94.25      299.780243 -2.213564 -0.340872  296.740417   
                    94.50      299.781006 -1.768703 -0.279048  296.868286   
                    94.75      299.840515 -1.137167 -0.332458  296.719696   
                    95.00      299.997192 -0.240758 -0.114535  296.810577   

                                     tp  
valid_time latitude longitude            
2020-01-01 35.0     65.00      0.000635  
                    65.25      0.000564  
                    65.50      0.000510  
                    65.75      0.000557  
                    66.00      0.000766  
...                                 ...  
2024-12-31 5.0      94.00      0.000742  
                    94.25      0.000671  
                    94.50      0.000475  
                    94.75      0.000276  
                    95.00      0.000079  

[26749107 rows x 5 columns]

In [7]:
combined_df.to_csv("combined_2020-2024_daily.csv")